# Übersetzung NL → DE — Runner

Code kommt per `git` in die VM, gearbeitet wird im Drive-Projektordner.
Jeder fertige Chunk liegt sofort dauerhaft in Drive.

**Ein Abbruch ist ein Nicht-Ereignis.** Trennt sich die VM, genügt ein
erneuter Klick auf Zelle 1 — der Resume zählt die Dateien in `teile/`
und setzt am nächsten offenen Chunk fort.

Beim ersten Start einer Sitzung in dieser Reihenfolge: **1 → 2 → 3**.
Danach genügt Zelle 1.

Secrets im Colab-Reiter „Secrets" hinterlegen und dieser Sitzung
Zugriff geben: `ANTHROPIC_API_KEY` und `GoogleKI`.

## 1 · Vorbereiten und laufen

Mountet Drive, holt den Code, leert alte Modulimporte, lädt die
Secrets, wechselt in den Projektordner und startet den Lauf im
Vordergrund. Die laufende Fortschrittsausgabe hält die Sitzung
nebenbei wach — sie ist kein Beiwerk.

Meldet diese Zelle abweichende technische Einstellungen, erst
Zelle 2 ausführen und danach hierher zurück.

In [ ]:
PROJEKT = "/content/drive/MyDrive/uebersetzung/1919"
ZWEIG   = "main"

REPO = "https://github.com/iinoox-ai/Claude-Code-Translate-.git"
CODE = "/content/Claude-Code-Translate-"

import os, subprocess, sys
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--branch", ZWEIG, REPO, CODE],
                   check=True)

# Frisch geholter Code nuetzt nichts, solange der Kernel die alten
# Module festhaelt: "import" ist dann ein No-op. Erst leeren, dann neu
# importieren — sonst laeuft die Zelle mit dem Stand vom letzten Start.
for _name, _modul in list(sys.modules.items()):
    if (getattr(_modul, "__file__", None) or "").startswith(CODE):
        del sys.modules[_name]
if CODE not in sys.path:
    sys.path.insert(0, CODE)

import colab_start
colab_start.vorbereiten(PROJEKT, code=CODE)
colab_start.lauf("pipeline.py", "run", code=CODE)

## 2 · Technische Einstellungen abgleichen

Die `projekt.json` im Drive-Ordner wird nie überschrieben — sie trägt
die kalibrierten Prüfgrenzen. Modellnamen und Effort gehören dagegen
zum Code und müssen nachgezogen werden, wenn sich im Repo etwas
geändert hat.

Läuft als eigener Prozess und sieht deshalb immer den Code auf der
Platte, nie einen veralteten Import.

In [ ]:
# ohne "--uebernehmen" nur anzeigen
colab_start.lauf("pipeline.py", "technik", "--uebernehmen", code=CODE)

## 3 · Verifikation — einmalig vor dem ersten Volllauf

Prüft die Schreibsemantik des Drive-Mounts, pingt jedes konfigurierte
Modell, übersetzt einen Kurz-Chunk je Anbieter mit Ausweis der
Token-Usage, belegt die Sampling-Doktrin an der echten API und
gleicht die Google-Tarife gegen die Preisseite ab.

Kostet wenige Cent. **Nach Zelle 2 ausführen** — sonst prüft sie
womöglich Modellnamen, die das Repo längst korrigiert hat.

In [ ]:
colab_start.lauf("verifikation.py", code=CODE)

## 4 · Stand

Zeigt, welche Schritte fertig sind und wo der Chunkzähler steht.

Colab arbeitet Zellen nacheinander ab: Solange Zelle 1 läuft, wird
diese Zelle nur eingereiht, nicht ausgeführt. Den Chunkstand während
eines Laufs liest man deshalb direkt an der Ausgabe von Zelle 1 ab.

In [ ]:
colab_start.lauf("pipeline.py", "status", code=CODE)